# Audio to timestamped text

Pause the web app while running GPU notebooks. These cells use the same modules as FastAPI.


Source mapping: `lab_2_local.ipynb`, cells 13 and 15 (transcription setup/execution). `app.engines.Engines.transcribe` preserves the tiny.en model and GTX 1650-compatible int8_float32 GPU computation. FFmpeg normalizes incoming audio.


In [ ]:
from pathlib import Path
import sys
# Find the standalone project whether Jupyter starts at its root or notebooks/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "app/config.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from app.config import Settings
from app.processors import Processor
from app.files import atomic_text, save_json
settings = Settings(root=ROOT)
settings.initialize()
processor = Processor(settings, progress=print)



In [ ]:
source = next((ROOT / "input/audio").rglob("*.wav"), None)
if source is None:
    raise FileNotFoundError("Add a WAV under input/audio/ first.")
print(source)



Run real transcription. The result includes timestamps and backend execution reports. A full recording may take time. The speech model unloads before later visual stages.


In [ ]:
result = processor.process(source, "audio")
print(result.devices)



In [ ]:
# Write one text output with its original extension and relative path preserved.
relative = source.relative_to(ROOT / "input")
output = ROOT / "output" / (relative.as_posix() + ".txt")
atomic_text(output, result.text(relative.as_posix()))
save_json(ROOT / "runtime" / "notebook-result.json", {
    "source": str(relative), "warnings": result.warnings,
    "devices": result.devices, "details": result.details,
})
print(output)
print(output.read_text(encoding="utf-8")[:5000])

